In [ ]:
# Sprawdzanie CUDA:

import torch

print(torch.cuda.is_available())
if (torch.cuda.is_available()):
    print(torch.cuda.get_device_name(0))  # nazwa karty
    print(torch.version.cuda)
else:
    print("brak")

True
Tesla T4
12.8


In [ ]:
!pip install -q transformers==4.43.3 accelerate==0.33.0 safetensors sentencepiece "tokenizers>=0.19.0" faiss-cpu pymupdf sentence_transformers ipywidgets numpy scipy tqdm huggingface-hub hf_xet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 121.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 126.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 106.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 129.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>

In [ ]:
import sys
import os
import json
import torch
import faiss
import pymupdf
import numpy as np
from tqdm import tqdm

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, logging

# niech się tylko wyświetlają błędy
logging.set_verbosity_error()

model_id = "microsoft/Phi-3.5-mini-instruct"

# Model embeddingowy – zamienia tekst na wektory liczbowe (embeddingi)
# all-mpnet-base-v2 produkuje wektory o wymiarze 768
embedder = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

# Tokenizer rozbija tekst na tokeny (podjednostki słów) zrozumiałe dla modelu
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Ładowanie modelu językowego (LLM) – Phi-3.5-mini ma 3.8B parametrów
# device_map="auto" – automatycznie wybiera GPU jeśli dostępne, inaczej CPU
# torch_dtype=float16 – zmniejsza zużycie pamięci kosztem precyzji (tylko GPU)
# attn_implementation="eager" – standardowa implementacja uwagi (bez flash-attention)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="eager"
)

# Tryb ewaluacji – wyłącza dropout, model nie jest trenowany tylko używany
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
          (rotary_emb): Phi3LongRoPEScaledRotaryEmbedding()
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLU()
        )
        (input_layernorm): Phi3RMSNorm()
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
        (post_attention_layernorm): Phi3RMSNorm()
      )
    )
    (norm): Phi3RMSNorm()
  )
  (lm_head): Linear(in_features=3072, out

In [ ]:
# Tworzymy pusty indeks FAISS typu FlatL2 – przechowuje wektory i szuka po odległości euklidesowej
# get_embedding_dimension() zwraca wymiar wektorów modelu embeddingowego (768)
index = faiss.IndexFlatL2(embedder.get_embedding_dimension())

# Lista słowników przechowująca metadane każdego chunka (nazwa pliku, numer strony, tekst)
metadata = []

print('Number of chunks: ', index.ntotal) # 0

Number of chunks:  0


In [ ]:
class Utils:
    def __init__(self, embedding_model: SentenceTransformer=None, llm_model: AutoModelForCausalLM=None, llm_tokenizer: AutoTokenizer=None, index=None, metadata=None, chunk_size=512):
        self.embedding_model = embedding_model
        self.llm_model = llm_model
        self.llm_tokenizer = llm_tokenizer
        self.index = index
        self.metadata = metadata
        # Rozmiar chunka w znakach – określa jak długie będą fragmenty tekstu
        self.chunk_size = chunk_size

    def extract_text_from_pdf(self, pdf_path):
        """
        Extract text from PDF file. Returns a list of tuples (page_number, text).
        """
        text = []
        pdf_document = pymupdf.open(pdf_path)
        for page_num in range(len(pdf_document)):
            page = pdf_document.load_page(page_num)
            # Zamieniamy znaki nowej linii na spacje, żeby tekst był ciągły
            text.append((page_num, str(page.get_text()).replace("\n", " ")))
        return text

    def chunk_text(self, text: list[tuple[int, str]]):
        chunks = []
        for page_num, page_text in text:
            # Dzielimy tekst strony na fragmenty o długości chunk_size znaków
            page_chunks = [
                (page_num, page_text[i:i+self.chunk_size])
                for i in range(0, len(page_text), self.chunk_size)
            ]
            chunks.extend(page_chunks)
        return chunks

    def add_chunks_to_faiss(self, chunks, filename, db_loc="vec_db/"):
        for chunk_num, (page_number, chunk) in enumerate(tqdm(chunks, desc="Adding chunks to FAISS")):
            # Zamieniamy tekst chunka na wektor embeddingowy
            embeddings = self.embedding_model.encode(chunk, show_progress_bar=False)
            # Dodajemy wektor do indeksu FAISS
            self.index.add(np.array([embeddings]))
            # Zapisujemy metadane chunka – powiążemy je z wektorem przez pozycję w indeksie
            self.metadata.append({
                "filename": filename,
                "page_number": page_number,
                "chunk_num": chunk_num,
                "chunk": chunk
            })
        # Zapisujemy indeks FAISS na dysk, żeby nie trzeba było go odbudowywać przy każdym uruchomieniu
        faiss.write_index(self.index, db_loc + "vector_database.index")
        with open(db_loc + "metadata.json", "w") as file:
            json.dump(self.metadata, file)

    def process_file(self, file_path):
        """
        Process the file and add chunks to FAISS index
        """
        if file_path.endswith('.pdf'):
            text = self.extract_text_from_pdf(file_path)
        else:
            print(f"Unsupported file format, with extension: {os.path.splitext(file_path)[1]}")
            return 0

        chunks = self.chunk_text(text)
        self.add_chunks_to_faiss(chunks, filename=os.path.basename(file_path))
        return len(chunks)

    def answer_question(self, prompt_template="", query="", max_tokens=512, temp=0.7, k=5):
        # Zamieniamy pytanie użytkownika na wektor embeddingowy
        question_embedding = self.embedding_model.encode(query, show_progress_bar=False)

        # Szukamy k najbliższych wektorów w FAISS – D to odległości, I to indeksy znalezionych chunków
        D, I = self.index.search(np.array([question_embedding]), k)
        # Pobieramy metadane (tekst) znalezionych chunków
        chunks = [self.metadata[i] for i in I[0]]

        # Sklejamy teksty chunków w jeden blok kontekstu dla modelu
        context = ""
        for i, chunk in enumerate(chunks):
            context += f"{i+1}. {chunk['chunk']}\n"

        # Wstawiamy kontekst i pytanie do szablonu promptu
        prompt = prompt_template.format(context=context, query=query)

        # Budujemy historię rozmowy w formacie wymaganym przez Phi-3.5
        messages = [
            {
                "role": "system",
                "content": (
                    "Be helpful, straight to the point. "
                    "Use only context. Do not hallucinate."
                )
            },
            {"role": "user", "content": prompt},
        ]

        # Tokenizujemy całą rozmowę i formatujemy zgodnie z szablonem czatu modelu
        # return_tensors="pt" zwraca tensor PyTorch gotowy do przekazania do modelu
        input_ids = self.llm_tokenizer.apply_chat_template(
            conversation=messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(self.llm_model.device)  # przenosimy tensor na to samo urządzenie co model

        # Maska uwagi – jedynki oznaczają tokeny które model ma brać pod uwagę (wszystkie)
        attention_mask = torch.ones_like(input_ids)
        # Zapamiętujemy długość promptu, żeby później wyciąć tylko wygenerowaną odpowiedź
        input_len = input_ids.shape[1]

        # Generujemy odpowiedź – model dokańcza sekwencję tokenów za promptem
        outputs = self.llm_model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_tokens,  # maksymalna liczba nowych tokenów do wygenerowania
            temperature=temp,           # im niższa tym bardziej deterministyczna odpowiedź
            do_sample=True,             # losowe próbkowanie (wymagane gdy temperature != 1.0)
        )

        # Odcinamy tokeny promptu – zostawiamy tylko to co model wygenerował
        generated = outputs[0][input_len:]
        # Dekodujemy tokeny z powrotem na tekst
        answer = self.llm_tokenizer.decode(generated, skip_special_tokens=True)

        return answer, chunks

In [ ]:
# Tworzymy obiekt Utils łącząc wszystkie komponenty RAG w jednym miejscu
utils = Utils(
    embedder,   # model embeddingowy do zamiany tekstu na wektory
    model,      # LLM do generowania odpowiedzi
    tokenizer,  # tokenizer dla LLM
    index,      # indeks FAISS z wektorami chunków
    metadata,   # metadane chunków (tekst, strona, plik)
    chunk_size=512 # ilość treści w każdym fragmencie - zwiększamy by dać modelowi więcej treści gdy odmówi odpowiedzi
)

In [ ]:
import os
os.makedirs("vec_db", exist_ok=True)

In [ ]:
knowledge_dir = "/content/"
for file in os.listdir(knowledge_dir):
    if file.endswith(".pdf"):  # pomijamy inne pliki systemowe Colaba
        utils.process_file(knowledge_dir + file)

print('Number of chunks: ', index.ntotal)

Adding chunks to FAISS: 100%|██████████| 127/127 [00:02<00:00, 61.64it/s]
Adding chunks to FAISS: 0it [00:00, ?it/s]
Adding chunks to FAISS: 100%|██████████| 275/275 [00:03<00:00, 78.77it/s]

Number of chunks:  1032


In [ ]:
# przukładowe pytania dotyczace powyzszych dokumentow
questions = [
    "Dlaczego astronomia kosmiczna jest ważna dla współczesnej nauki?",
    "Jak atmosfera Ziemi wpływa na obserwacje astronomiczne?",
    "Jakie rodzaje promieniowania elektromagnetycznego są wykorzystywane w astronomii?",
    "Czym różni się teleskop optyczny od radioteleskopu?",
    "Jakie informacje o obiektach kosmicznych można uzyskać dzięki analizie widma?",
    "Jak astronomowie wykorzystują podczerwień do badania kosmosu?",
    "Czym jest interferometria w astronomii?",
    "Jakie odkrycia umożliwiły obserwacje w zakresie fal radiowych?",
    "Czym są egzoplanety?",
    "Jakie są główne metody wykrywania egzoplanet?",
    "Dlaczego wykrywanie małych egzoplanet jest trudniejsze niż dużych?",
    "Czym jest strefa zamieszkiwalna wokół gwiazdy?",
    "Jak astronomowie badają atmosfery egzoplanet?",
    "Jakie cechy planety mogą wskazywać na możliwość istnienia życia?",
    "Jakie typy egzoplanet odkryto do tej pory?",
    "Jakie przyszłe technologie mogą poprawić wykrywanie egzoplanet?",
    "Czym jest astroML?",
    "Jak machine learning jest wykorzystywany w astronomii?",
    "Jakie typy danych astronomicznych analizuje się za pomocą ML?",
    "Dlaczego astronomia generuje duże ilości danych?",
    "Jakie algorytmy klasyfikacji są stosowane w analizie danych astronomicznych?",
    "Na czym polega klasteryzacja obiektów astronomicznych?",
    "W jaki sposób wykrywa się anomalie w danych astronomicznych?",
    "Jakie znaczenie mają sieci neuronowe w analizie obrazów kosmosu?",
    "Jak działa analiza szeregów czasowych w obserwacjach astronomicznych?",
    "Czym różni się supervised learning od unsupervised learning w kontekście astronomii?",
    "Jak ML pomaga w klasyfikacji galaktyk?",
    "Czym zajmuje się astrochemia?",
    "Jak powstają cząsteczki w przestrzeni międzygwiazdowej?",
    "Jaką rolę odgrywa pył kosmiczny w formowaniu gwiazd i planet?",
    "Dlaczego lód międzygwiazdowy jest ważny dla chemii kosmosu?",
    "Jakie związki organiczne odkryto w obłokach molekularnych?",
    "Jakie procesy zachodzą w dyskach protoplanetarnych?",
    "Jakie znaczenie ma astrochemia dla badań nad pochodzeniem życia?",
    "Jakie techniki spektroskopowe stosuje się w astrochemii?",
    "Jakie są główne źródła pyłu międzygwiazdowego?",
    "Jak astrochemia pomaga badać atmosfery egzoplanet?",
]

In [ ]:
from IPython.display import display, Markdown

prompt_template ="""Based on the following context items, please answer the query.
Give yourself room to think by extracting relevant passages from the context before answering the query.
Don't return the thinking, only return the answer.
Answer in Polish language only.
Use the following examples as reference for the ideal answer style.
Example 1:
Pytanie: Dlaczego Księżyc zawsze pokazuje tę samą stronę Ziemi?
Księżyc pokazuje Ziemi zawsze tę samą stronę, ponieważ jest związany pływowo z Ziemią. Oznacza to, że jego czas obrotu wokół własnej osi jest równy czasowi obiegu wokół Ziemi (około 27,3 dnia). W wyniku działania sił grawitacyjnych Ziemi rotacja Księżyca została w przeszłości spowolniona aż do osiągnięcia tego stanu równowagi.
Now use the following context items to answer this one user query only:
{context}
Relevant passages: <extract relevant passages from the context here>
Main User Query: {query}
Answer:\n"""

# Wybieramy losowo zapytanie z listy
random_query = np.random.choice(questions)

response, chunks = utils.answer_question(
        prompt_template=prompt_template,
        query=random_query,
        max_tokens=512,
        temp=0.1
)

display(Markdown(f"**Pytanie:** {random_query}"))
display(Markdown(f"**Odpowiedź:**\n\n{response}"))
display(Markdown("---\n**Źródła:**"))
for i, chunk in enumerate(chunks):
    excerpt = chunk['chunk'][:200].strip() + "..."
    display(Markdown(
        f"**[{i+1}]** `{chunk['filename']}` — strona {chunk['page_number'] + 1}\n\n"
        f"> {excerpt}"
    ))

**Pytanie:** Jakie są główne metody wykrywania egzoplanet?

**Odpowiedź:**

Główne metody wykrywania egzoplanet obejmują:

1. Wykrywanie przez obrazowanie: Podsumowuje się w referencach 3 i 4, gdzie wspomina się o Darwin misji dla eksponerowania egzoplanet z powodzeniem z powrotem do kosmosu. Ten metoda obejmuje fotografowanie egzoplanet od spaciego poziomu.

2. Wykrywanie przez microlensing: Podsumowuje się w referencie 5, gdzie wspomina się o koncepcji microlensing planetarnych, która polega na obserwowaniu zwiększenia i zmniejszenia obrazów starego gwiazdy, gdy egzoplanet przechodzi przez obszar grawitacyjny swojego hosta.

3. Wykrywanie przez transzyt: Podsumowuje się w referencie 5, gdzie wspomina się o transzyt, który obejmuje obserwację zmian w światło starego gwiazdy, gdy egzoplanet przechodzi przez jego hosta, zanim zostaje zobaczyć na niej.

Tak więc główne metody wykrywania egzoplanet to obrazowanie, microlensing i transzyt.

---
**Źródła:**

**[1]** `Exoplanets.pdf` — strona 7

> , A. L´eger et al., Experimental Astronomy 23, 435–461 (2009). A summary of the scientiﬁc and design goals of the Darwin mission for exoplanet imaging from space, now in hibernation. (I) 56. Ref. 3, C...

**[2]** `Exoplanets.pdf` — strona 7

> , A. L´eger et al., Experimental Astronomy 23, 435–461 (2009). A summary of the scientiﬁc and design goals of the Darwin mission for exoplanet imaging from space, now in hibernation. (I) 56. Ref. 3, C...

**[3]** `Exoplanets.pdf` — strona 7

> , A. L´eger et al., Experimental Astronomy 23, 435–461 (2009). A summary of the scientiﬁc and design goals of the Darwin mission for exoplanet imaging from space, now in hibernation. (I) 56. Ref. 3, C...

**[4]** `Exoplanets.pdf` — strona 7

> , A. L´eger et al., Experimental Astronomy 23, 435–461 (2009). A summary of the scientiﬁc and design goals of the Darwin mission for exoplanet imaging from space, now in hibernation. (I) 56. Ref. 3, C...

**[5]** `Exoplanets.pdf` — strona 6

> , ARA&A 50, 411–453 (2012). Review of the concepts of microlensing planet searches and their practical application. (I) 34. Ref. 3, Chapter 5 provides details of the principles, discov- eries, and bib...

In [ ]:
import subprocess, time

# 1) Zależność wymagana przez instalator Ollama
!apt-get install -y zstd

# 2) Instalacja Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 3) Serwer w tle
subprocess.Popen(["/usr/local/bin/ollama", "serve"],
                 stdout=subprocess.DEVNULL,
                 stderr=subprocess.DEVNULL)
print("Czekam aż serwer wstanie...")
time.sleep(10)
print("✅ Ollama działa")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 51 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (1,668 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122363 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Add

In [ ]:
# qwen2.5:3b jest lżejszy niż 7b – lepszy wybór dla Colab
os.system("ollama pull qwen2.5:3b")

0

In [ ]:
!pip install -q ollama

In [ ]:
import ollama as ollama_client

def answer_question_ollama(self, prompt_template="", query="", max_tokens=512, temp=0.1, k=5, ollama_model="qwen2.5:3b"):
    question_embedding = self.embedding_model.encode(query, show_progress_bar=False)
    D, I = self.index.search(np.array([question_embedding]), k)
    chunks = [self.metadata[i] for i in I[0]]

    context = ""
    for i, chunk in enumerate(chunks):
        context += f"{i+1}. {chunk['chunk']}\n"

    prompt = prompt_template.format(context=context, query=query)

    response = ollama_client.chat(
        model=ollama_model,
        messages=[
            {
                "role": "system",
                "content": "Be helpful, straight to the point. Use only context. Do not hallucinate."
            },
            {"role": "user", "content": prompt}
        ],
        options={"temperature": temp, "num_predict": max_tokens}
    )

    answer = response["message"]["content"].strip()
    return answer, chunks

# Podpinamy nową metodę do istniejącego obiektu utils
import types
utils.answer_question = types.MethodType(answer_question_ollama, utils)


In [ ]:
from IPython.display import display, Markdown

random_query = np.random.choice(questions)

response, chunks = utils.answer_question(
    prompt_template=prompt_template,
    query=random_query,
    max_tokens=512,
    temp=0.1,
    ollama_model="qwen2.5:3b"
)

display(Markdown(f"**Pytanie:** {random_query}"))
display(Markdown(f"**Odpowiedź:**\n\n{response}"))
display(Markdown("---\n**Źródła:**"))
for i, chunk in enumerate(chunks):
    excerpt = chunk['chunk'][:200].strip() + "..."
    display(Markdown(
        f"**[{i+1}]** `{chunk['filename']}` — strona {chunk['page_number'] + 1}\n\n"
        f"> {excerpt}"
    ))

**Pytanie:** Jakie typy egzoplanet odkryto do tej pory?

**Odpowiedź:**

Do tej pory odkryto różne typy egzoplanet, ale niektóre z informacji są nieaktualne lub brakują. Istnieje up-to-date katalog i bibliografia potwierdzonych exoplanet wykrytych przez wszystkie metody, a także oddzielna lista niepotwierdzonych czy anulowanych planet.

---
**Źródła:**

**[1]** `Exoplanets.pdf` — strona 1

> intains an up-to-date catalogue and bibliography of conﬁrmed exoplanets detected by all methods, and a separate list of unconﬁrmed or retracted planets. arXiv:1311.2521v2  [astro-ph.EP]  12 Nov 2013...

**[2]** `Exoplanets.pdf` — strona 1

> intains an up-to-date catalogue and bibliography of conﬁrmed exoplanets detected by all methods, and a separate list of unconﬁrmed or retracted planets. arXiv:1311.2521v2  [astro-ph.EP]  12 Nov 2013...

**[3]** `Exoplanets.pdf` — strona 1

> intains an up-to-date catalogue and bibliography of conﬁrmed exoplanets detected by all methods, and a separate list of unconﬁrmed or retracted planets. arXiv:1311.2521v2  [astro-ph.EP]  12 Nov 2013...

**[4]** `Exoplanets.pdf` — strona 1

> intains an up-to-date catalogue and bibliography of conﬁrmed exoplanets detected by all methods, and a separate list of unconﬁrmed or retracted planets. arXiv:1311.2521v2  [astro-ph.EP]  12 Nov 2013...

**[5]** `Exoplanets.pdf` — strona 11

> 11 star and planet formation, and exoplanet detection, there- after focusing on the development and search for life, and detailed considerations of the habitable zone. (I) 103. “The anthropic principl...